# Drivers 00 — Base table
Builds `driver_data/base_transects.parquet` (228,538 NZCCD transects, key
`UniqueID` int64) from `rates_with_timeseries.parquet` + the published
shapefile's `TCD`, plus the `(UniqueID, Date) → uncertainty` lookup used by the
epoch module, and verifies the data conventions.

**Conventions established here** (full details in `driver_data/qa/CONVENTIONS.md`):
- positive WLR/NSM = accretion (verified empirically against the DEM);
- the national alongshore axis comes from the UniqueID itself
  (`chain_m = (UniqueID % 1e9)/100`, 10 m steps) — the published `TCD`
  resets per mapped section and is unusable for national ordering;
- transect vertex order is not uniform; the landward end is determined
  per transect in Drivers_04.

In [1]:
from drivers import common
from drivers.base_build import build_base, build_uncy_lookup, check_landward_convention
import pandas as pd

if not (common.DRIVER_DATA / "base_transects.parquet").exists():
    build_base()
if not (common.DRIVER_DATA / "uncy_lookup.parquet").exists():
    build_uncy_lookup()

base = common.load_base()
print(f"{len(base):,} transects, {base.UniqueID.is_unique=}")
base.head()

228,538 transects, base.UniqueID.is_unique=True


,UniqueID,Region,Start_date,End_date,Duration,ShrCount,NSM,SCE,EPR,EPRunc,...,WSE,WR2,lat,lon,x2193,y2193,TCD,island,chain_m,block_id
0,203054437032,Southland,1958-02-10,2007-02-05,49.0,3.0,28.13,28.13,0.57,0.12,...,1.41,0.96,-47.049447,167.704192,1.197796e+06,4.775717e+06,0.0,203,544370.32,2030000272
1,203054438164,Southland,1958-02-10,2007-02-05,49.0,3.0,29.60,29.60,0.60,0.12,...,1.45,0.96,-47.049464,167.704338,1.197807e+06,4.775716e+06,10.0,203,544381.64,2030000272
2,203054439252,Southland,1958-02-10,2007-02-05,49.0,3.0,30.68,30.68,0.63,0.12,...,1.15,0.97,-47.049473,167.704481,1.197818e+06,4.775716e+06,20.0,203,544392.52,2030000272
3,203054440300,Southland,1958-02-10,2007-02-05,49.0,3.0,30.80,30.80,0.63,0.12,...,0.96,0.98,-47.049471,167.704618,1.197828e+06,4.775717e+06,30.0,203,544403.00,2030000272
4,203054441325,Southland,1958-02-10,2007-02-05,49.0,3.0,31.66,31.66,0.65,0.12,...,0.80,0.99,-47.049463,167.704752,1.197838e+06,4.775718e+06,40.0,203,544413.25,2030000272


In [2]:
# sanity: chainage steps at ~10 m, TCD fully joined
step = base.sort_values(["island", "chain_m"]).groupby("island").chain_m.diff()
print("median alongshore step (m):", step.median())
print("TCD missing:", base.TCD.isna().sum())
base.groupby("Region").size().sort_values(ascending=False)

median alongshore step (m): 10.010000000009313
TCD missing: 0


Region
Northland        37267
Canterbury       21875
West Coast       20987
Taranaki         17047
Waikato          16587
Otago            15424
Bay of Plenty    13902
Hawkes Bay       13494
Southland        12380
Auckland         11488
Wellington       11113
Manawatu         10441
Tasman           10101
Gisborne          8313
Marlborough       5083
Nelson            3036
dtype: int64

Conventions file:

In [3]:
print(open(common.QA_DIR / "CONVENTIONS.md").read())

# NZCCD conventions (empirically verified 2026-08-07)

- **Sign**: positive WLR/NSM/EPR/LRR = accretion (seaward movement); negative =
  erosion. Verified by mapping earliest-shoreline positions back onto transect
  geometry: NSM > +50 m transects have their earliest shoreline on land today
  (median elev +5.05 m, 92% > 2 m); NSM < −50 m transects at sea level
  (median −0.16 m, 11% > 2 m). n=300 each.
- **Distance origin**: DSAS.ipynb measures Distance from `shapely.get_point(t, -1)`
  (last vertex). Transect vertex order is NOT uniform nationally: the robust
  per-transect determination in d04 (valid-DEM fraction + mean elevation of the
  two sides adjacent to the latest shoreline) finds the LAST vertex landward for
  86% of transects and the FIRST for 14%. Never assume a fixed landward end.
  (A naive endpoint-validity test misleads because some LiDAR surveys map water
  as valid ~0 m elevation.)
- **Alongshore axis**: published `TCD` resets per mapped section (max ~67 km).
  The na